# BioEmu Conformational Ensemble Sampling

![BioEmu Conformational Ensemble Sampling](https://proto-bio.github.io/proto-assets/images/tool/bioemu/hero.png)

This notebook demonstrates conformational ensemble sampling using BioEmu. We generate a set of backbone conformations for a short protein sequence, inspect the resulting ensemble, and demonstrate batch sampling across multiple complexes in a single call.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("bioemu")
display_overview("bioemu")
display_docs_section("bioemu", "Background")

# BioEmu

BioEmu is a generative deep learning model from Microsoft Research that samples the [equilibrium](https://en.wikipedia.org/wiki/Boltzmann_distribution) conformational ensemble of a protein from its sequence. Rather than predicting a single static structure, it draws many independent backbone conformations that approximate the distribution of states a protein populates in solution, providing a fast alternative to [molecular dynamics](https://en.wikipedia.org/wiki/Molecular_dynamics) for surveying [conformational flexibility](https://en.wikipedia.org/wiki/Protein_dynamics). This tool implementation exposes a single operation that samples ensembles for one or more monomeric protein sequences.

A protein in solution is not a single fixed shape. It fluctuates among many [conformations](https://en.wikipedia.org/wiki/Protein_dynamics), and this flexibility underlies catalysis, [allosteric regulation](https://en.wikipedia.org/wiki/Allosteric_regulation), and molecular recognition. Characterizing this ensemble experimentally is difficult, and physically simulating it with [molecular dynamics](https://en.wikipedia.org/wiki/Molecular_dynamics) is accurate but computationally demanding, since the timescales of biologically relevant motions can require enormous amounts of simulation.

BioEmu ([Lewis et al., 2025](https://doi.org/10.1126/science.adv9817)) approaches the problem with a [diffusion-based generative model](https://en.wikipedia.org/wiki/Diffusion_model) that learns to emulate protein equilibrium ensembles directly. Starting from noise, the model iteratively denoises protein backbone coordinates conditioned on a sequence embedding, producing thousands of statistically independent structures per hour on a single [graphics processing unit](https://en.wikipedia.org/wiki/Graphics_processing_unit). The published model was trained on a large corpus of molecular dynamics simulation alongside static structures and experimental protein stability measurements, and it reproduces functional motions such as cryptic pocket formation, local unfolding, and domain rearrangements while approximating relative free energies. The conditioning sequence embedding is derived from a [multiple sequence alignment](https://en.wikipedia.org/wiki/Multiple_sequence_alignment), so each sequence is first searched against sequence databases to assemble its alignment.

## Available tools

In [2]:
display_available_tools("bioemu")

- **`run_bioemu()`** — Protein conformational ensemble sampling using BioEmu

### `run_bioemu`

BioEmu generates protein conformational ensembles by sampling from a learned distribution over backbone conformations. It accepts one or more single-chain protein complexes and returns an ensemble of backbone structures for each, representing the thermodynamic diversity of conformations that the protein is likely to adopt. BioEmu always runs MSA search via ColabFold before inference, unless pre-computed MSAs are already attached to the input. Multiple complexes can be submitted in a single call; the model processes them together and returns one ensemble per input.

In [3]:
from proto_tools import (
    BioEmuConfig,
    BioEmuInput,
    Complex,
    run_bioemu,
)

In [4]:
# Display input docs
display_api_reference("bioemu", "input", "run_bioemu")

# Chignolin (CLN025) is a 10-residue fast-folding beta-hairpin, a classic benchmark
# for conformational ensemble sampling.
complex_ = Complex(
    chains=[{"sequence": "YYDPETGTWY", "entity_type": "protein"}]
)

inputs = BioEmuInput(complexes=[complex_])

**Input** — `BioEmuInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>complexes</code> | <code>list[Complex]</code> | required | List of complexes to predict structure for containing chains and entity types. |
| <code>msas</code> | <code>list[ComplexMSAs] &#124; None</code> | <code>None</code> | Per-complex MSAs; a bare dict[int, MSA] is coerced to an unpaired ComplexMSAs. |

In [5]:
# Display config docs
display_api_reference("bioemu", "config", "run_bioemu")

# Sample 50 conformations using the v1.1 model with quality filtering enabled
config = BioEmuConfig(
    num_samples=50,
    model_name="bioemu-v1.1",
    filter_samples=True,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `BioEmuConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on (e.g., 'cuda', 'cpu') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>include_pae_matrix</code> | <code>bool</code> | <code>False</code> | Unused by BioEmu (inherited from StructurePredictionConfig) |
| <code>num_samples</code> | <code>int</code> | <code>500</code> | Number of conformations to sample per sequence |
| <code>model_name</code> | <code>Literal['bioemu-v1.0', 'bioemu-v1.1', 'bioemu-v1.2']</code> | <code>'bioemu-v1.1'</code> | BioEmu checkpoint (v1.1 = Science paper; v1.2 = extended MD + folding-FE training) |
| <code>filter_samples</code> | <code>bool</code> | <code>True</code> | Drop unphysical samples (steric clashes, chain discontinuities) |
| <code>batch_size</code> | <code>int</code> | <code>10</code> | Batch size at L=100; effective batch scales as batch_size * (100/L)^2 |
| <code>denoiser_type</code> | <code>Literal['dpm', 'heun']</code> | <code>'dpm'</code> | Diffusion sampler algorithm (dpm = 50 deterministic steps; heun = stochastic) |
| <code>denoiser_config</code> | <code>str &#124; None</code> | <code>None</code> | Path to a custom denoiser/steering YAML; overrides denoiser_type when set |
| <code>msa_host_url</code> | <code>str &#124; None</code> | <code>None</code> | Override the ColabFold MMseqs2 MSA server URL |
| <code>cache_embeds_dir</code> | <code>str &#124; None</code> | <code>None</code> | Directory to cache MSA embeddings across runs |
| <code>cache_so3_dir</code> | <code>str &#124; None</code> | <code>None</code> | Directory to cache SO3 precomputations across runs |
| <code>output_dir</code> | <code>str &#124; None</code> | <code>None</code> | Optional directory for raw BioEmu output files |
| <code>msa_search_config</code> | <code>Mmseqs2HomologySearchConfig &#124; None</code> | <code>None</code> | Configuration for MMseqs2 homology search (MSA generation); None uses defaults |

In [6]:
# Run the sampling
single_result = run_bioemu(inputs, config)

Running run_mmseqs2_homology_search [00:00]

Starting worker


Running bioemu


In [7]:
# Display output docs and inspect results
display_api_reference("bioemu", "output", "run_bioemu")

print(f"Ensembles: {len(single_result.ensembles)}")
print(f"Conformations: {len(single_result.ensembles[0].structures)}")

**Output** — `BioEmuOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>ensembles</code> | <code>list[StructureEnsemble]</code> | required | Generated protein conformational ensembles |

Ensembles: 1
Conformations: 50


### Batch sampling across multiple complexes

Multiple complexes can be submitted in a single call, which is more efficient than making separate calls because MSA search and GPU allocation happen only once. Each complex produces its own ensemble in the output.

In [8]:
from proto_tools import StructureMetricsConfig, StructureMetricsInput, run_structure_metrics

# Two real fast-folding mini-proteins with well-characterized conformational dynamics.
chignolin = Complex(chains=[{"sequence": "YYDPETGTWY", "entity_type": "protein"}])  # CLN025 beta-hairpin
trp_cage = Complex(chains=[{"sequence": "NLYIQWLKDGGPSSGRPPPS", "entity_type": "protein"}])  # Trp-cage mini-protein

batch_inputs = BioEmuInput(complexes=[chignolin, trp_cage])
batch_config = BioEmuConfig(num_samples=20, verbose=False)
batch_result = run_bioemu(batch_inputs, batch_config)

# Quantify conformational diversity via the radius-of-gyration spread across the ensemble.
for idx, ensemble in enumerate(batch_result.ensembles):
    metrics = run_structure_metrics(
        StructureMetricsInput(structures=ensemble.structures), StructureMetricsConfig()
    )
    rgs = [m.gyration_radius for m in metrics.metrics]
    print(
        f"Complex {idx}: {len(ensemble.structures)} conformations, "
        f"Rg {min(rgs):.1f}-{max(rgs):.1f} A (spread {max(rgs) - min(rgs):.2f} A)"
    )

Running run_mmseqs2_homology_search [00:00]

Starting worker


Running bioemu


Complex 0: 18 conformations, Rg 4.9-6.8 A (spread 1.90 A)


Complex 1: 18 conformations, Rg 8.6-15.9 A (spread 7.34 A)


## Export Results

Ensemble outputs can be saved in two formats. PDB format produces a directory tree with one PDB file per conformation, organized by ensemble, which is useful for visualization or as input to downstream tools. JSON format produces a single file containing all conformations, which is convenient for programmatic analysis.

In [9]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

batch_result.export(name="bioemu_batch", export_path=output_dir, file_format="json")
print("Batch ensembles exported to example_output/bioemu_batch.json")

Batch ensembles exported to example_output/bioemu_batch.json
